### Import depedencies

Superlinked loads `config.yaml` from the **current working directory** (see `config.yaml` at the repo root and a copy under `notebooks/`). `framework.APP_ID` sets the Qdrant collection name (`yelp-businesses-collection-00`). Restart the kernel after editing the YAML.

In [29]:
from qdrant_client import QdrantClient
import pandas as pd
import openai
from superlinked import framework as sl
import ast
import json
import re
from typing import Any, Optional
import pandas as pd
from api.core.config import config # run uv pip install -e . api folder level to set it as package in ediatble mode
from collections import namedtuple
import instructor
from instructor.core.exceptions import InstructorRetryException
instructor.InstructorRetryException = InstructorRetryException

# from superlinked_app.index import index
# from superlinked_app.query import query
# from superlinked_app.api import vector_database

### Read a sample dataset with Yelp business data

In [30]:
df_businesses = pd.read_json("../data/raw/yelp_academic_dataset_business_restaurants_with_hours_sample_1000.json", lines =True)

In [31]:
df_businesses.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,cHZ1FUswyko119EkV_9aNQ,Upper Crust Pizza,1909 E Grant Rd,Tucson,AZ,85719,32.250429,-110.943524,3.5,180,1,"{'BikeParking': 'True', 'RestaurantsPriceRange...","Salad, Pizza, Chicken Wings, Restaurants","{'Monday': '11:0-21:0', 'Tuesday': '11:0-21:0'..."
1,goTAL_3Ns-bsMgfGmKDq0A,Spitale's Deli & Catering,3309 Division St,Metairie,LA,70002,30.004793,-90.164386,4.5,53,1,"{'RestaurantsPriceRange2': '1', 'RestaurantsRe...","American (New), Restaurants, Local Flavor, Delis","{'Monday': '9:0-16:0', 'Tuesday': '10:30-20:0'..."
2,L1TzHZ8OXfVXTfyzidstzg,Snooze An AM Eatery,"2500 E Grant Rd, Ste 182",Tucson,AZ,85716,32.250337,-110.935414,4.5,253,1,"{'Caters': 'True', 'OutdoorSeating': 'True', '...","Breakfast & Brunch, American (Traditional), Sa...","{'Monday': '0:0-0:0', 'Tuesday': '6:30-14:30',..."
3,DfkO7MHuy2wxgeUZ2-o3BA,Subway,204 St Charles Ave,New Orleans,LA,70130,29.951985,-90.069679,3.0,8,0,"{'RestaurantsTakeOut': 'True', 'RestaurantsDel...","Restaurants, Sandwiches, Fast Food","{'Monday': '7:0-17:0', 'Tuesday': '7:0-17:0', ..."
4,ByKCIpZWHYUTuCJcoZ-ONQ,Giovanni's,6583 Pardall Rd,Goleta,CA,93117,34.412901,-119.857810,4.0,87,0,"{'RestaurantsTakeOut': 'True', 'RestaurantsGoo...","Salad, Sandwiches, Restaurants, Pizza, Italian","{'Monday': '11:0-20:0', 'Tuesday': '11:0-20:0'..."


In [32]:
df_businesses['is_open'].value_counts()

is_open
1    725
0    275
Name: count, dtype: int64

### Defining the Superlinked schema and vector space for queriable dimensions

In [ ]:
# ----------------------------
# Helpers (normalize Yelp fields)
# ----------------------------

_TRUE_SET = {"True","true", "1", "yes", "y", "t"}
_FALSE_SET = {"False","false", "0", "no", "n", "f"}

def parse_bool(v: Any) -> Optional[bool]:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in _TRUE_SET:
        return True
    if s in _FALSE_SET:
        return False
    return None


def bool_for_index(v: Any) -> bool:
    """Yelp omits many attributes; Index fields must be non-null on ingest."""
    b = parse_bool(v)
    return False if b is None else b


def parse_int(v: Any) -> Optional[int]:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    try:
        return int(str(v).strip())
    except Exception:
        return None

_wifi_pat = re.compile(r"^u'(.+)'$")  # converts "u'no'" -> "no"
def normalize_wifi(v: Any) -> Optional[str]:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip()
    m = _wifi_pat.match(s)
    if m:
        s = m.group(1)
    s = s.strip().strip('"').strip("'").lower()
    if s in {"no", "free", "paid"}:
        return s
    return "unknown"

def parse_categories_list(v: Any) -> list[str]:
    # Yelp "categories" commonly comes as "A, B, C"
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    return [c.strip() for c in s.split(",") if c.strip()]

def categories_text_from_list(cats: list[str]) -> str:
    # a single string for TextSimilaritySpace input
    return ", ".join(cats)

def parse_business_parking(v: Any) -> dict:
    # Yelp often stores this as a stringified python dict
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return {}
    if isinstance(v, dict):
        return v
    s = str(v).strip()
    if not s:
        return {}
    try:
        parsed = ast.literal_eval(s)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def parse_time_to_minutes(hhmm: str) -> Optional[int]:
    # handles "8:0" or "08:00"
    if hhmm is None:
        return None
    s = str(hhmm).strip()
    if not s:
        return None
    parts = s.split(":")
    if len(parts) != 2:
        return None
    try:
        h = int(parts[0])
        m = int(parts[1])
        if 0 <= h <= 24 and 0 <= m < 60:
            return h * 60 + m
    except Exception:
        return None
    return None

def parse_hours_range(v: Any) -> tuple[Optional[int], Optional[int], Optional[bool]]:
    # e.g. "8:0-22:0" -> (480, 1320, overnight=False)
    if v is None:
        return (None, None, None)
    s = str(v).strip()
    if not s or s == "0:0-0:0":
        return (None, None, None)
    if "-" not in s:
        return (None, None, None)
    start_s, end_s = [p.strip() for p in s.split("-", 1)]
    start = parse_time_to_minutes(start_s)
    end = parse_time_to_minutes(end_s)
    if start is None or end is None:
        return (None, None, None)
    overnight = end < start
    if overnight:
        end = end + 1440  # represent "next day" close
    return (start, end, overnight)


def minutes_to_hhmm(m: Any, *, zero_display: str = "—") -> str:
    """Format minutes since midnight for display (allows hours > 23 after overnight shift)."""
    if m is None or (isinstance(m, float) and pd.isna(m)):
        return zero_display
    try:
        mi = int(m)
    except (TypeError, ValueError):
        return zero_display
    if mi == 0:
        return zero_display
    h, mm = divmod(mi, 60)
    return f"{h:d}:{mm:02d}"


_MINUTE_TIME_COLUMNS = tuple(
    f"{day}_{kind}" for day in ("mon", "tue", "wed", "thu", "fri", "sat", "sun") for kind in ("open", "close")
)


def format_minute_columns_to_hhmm(df: pd.DataFrame) -> pd.DataFrame:
    """Convert indexed weekday open/close minute fields to hh:mm strings for display."""
    out = df.copy()
    for col in _MINUTE_TIME_COLUMNS:
        if col in out.columns:
            out[col] = out[col].apply(minutes_to_hhmm)
    return out


# ----------------------------
# Superlinked schema — same fields / order as df_businesses (Yelp business JSON).
# attributes & hours are dicts in pandas; indexed as stable JSON strings for Superlinked.
# `category_tags` = parsed Yelp labels (exact strings for filters); `categories_text` = joined for embeddings only.
# ----------------------------

class Business(sl.Schema):
    business_id: sl.IdField
    name: sl.String
    address: sl.String
    city: sl.String
    state: sl.String
    postal_code: sl.String
    latitude: sl.Float
    longitude: sl.Float
    stars: sl.Float
    review_count: sl.Integer
    is_open: sl.Boolean
    is_open_i: sl.Integer  # 1/0 mirror for queries + NLQ (bool params are not NLQ-supported)
    attributes: sl.String
    # Parsed from Yelp `attributes` dict; use *_i (0/1) so NLQ can fill int params (bool is unsupported).
    bike_parking_i: sl.Integer
    accepts_credit_cards_i: sl.Integer
    price_range: sl.Integer
    wifi: sl.String
    parking_garage_i: sl.Integer
    parking_street_i: sl.Integer
    parking_validated_i: sl.Integer
    parking_lot_i: sl.Integer
    parking_valet_i: sl.Integer
    wheelchair_accessible_i: sl.Integer
    happy_hour_i: sl.Integer
    outdoor_seating_i: sl.Integer
    has_tv_i: sl.Integer
    takes_out_i: sl.Integer
    delivery_i: sl.Integer
    reservations_i: sl.Integer
    dogs_allowed_i: sl.Integer
    by_appointment_only_i: sl.Integer
    category_tags: sl.StringList  # Yelp labels (exact strings) for contains_all filter
    categories_text: sl.String  # comma-joined; TextSimilaritySpace input only
    hours: sl.String
    # Parsed from Yelp `hours` dict (minutes since midnight; close may exceed 1440 if overnight)
    mon_open: sl.Integer
    mon_close: sl.Integer
    tue_open: sl.Integer
    tue_close: sl.Integer
    wed_open: sl.Integer
    wed_close: sl.Integer
    thu_open: sl.Integer
    thu_close: sl.Integer
    fri_open: sl.Integer
    fri_close: sl.Integer
    sat_open: sl.Integer
    sat_close: sl.Integer
    sun_open: sl.Integer
    sun_close: sl.Integer


business = Business()


# ----------------------------
# DataFrame -> schema mapping
# ----------------------------

def get_attr(row: pd.Series, key: str) -> Any:
    attrs = row.get("attributes")
    if isinstance(attrs, dict):
        return attrs.get(key)
    return None


def get_hours(row: pd.Series, day: str) -> Any:
    hours = row.get("hours")
    if isinstance(hours, dict):
        return hours.get(day)
    return None


def int01(v: Any) -> int:
    return 1 if bool_for_index(v) else 0


# DataFrameParser mapping values must be column names (str). Callable values are not
# row-wise transforms: pandas treats a callable indexer as df.apply(lambda), so the
# callable receives the whole DataFrame. Build a flat frame whose columns match schema
# field names, then use the default parser (no mapping).


def business_row_for_index(row: pd.Series) -> pd.Series:
    def _str_cell(v: Any) -> str:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ""
        return str(v).strip()

    def _float_cell(v: Any, default: float = 0.0) -> float:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return default
        return float(v)

    attrs_raw = row.get("attributes")
    attributes_s = (
        json.dumps(attrs_raw, sort_keys=True, default=str) if isinstance(attrs_raw, dict) else "{}"
    )
    park = parse_business_parking(get_attr(row, "BusinessParking"))
    hours_raw = row.get("hours")
    hours_s = json.dumps(hours_raw, sort_keys=True, default=str) if isinstance(hours_raw, dict) else "{}"
    is_open_b = bool(int(row["is_open"])) if pd.notna(row.get("is_open")) else False

    def oc(day: str) -> tuple[int, int]:
        o, c, _ = parse_hours_range(get_hours(row, day))
        return (o or 0, c or 0)

    mo_o, mo_c = oc("Monday")
    tu_o, tu_c = oc("Tuesday")
    we_o, we_c = oc("Wednesday")
    th_o, th_c = oc("Thursday")
    fr_o, fr_c = oc("Friday")
    sa_o, sa_c = oc("Saturday")
    su_o, su_c = oc("Sunday")
    cats = parse_categories_list(row.get("categories"))

    return pd.Series(
        {
            "business_id": row["business_id"],
            "name": _str_cell(row.get("name")),
            "address": _str_cell(row.get("address")),
            "city": _str_cell(row.get("city")),
            "state": _str_cell(row.get("state")),
            "postal_code": _str_cell(row.get("postal_code")),
            "latitude": _float_cell(row.get("latitude")),
            "longitude": _float_cell(row.get("longitude")),
            "stars": _float_cell(row.get("stars")),
            "review_count": int(row["review_count"]) if pd.notna(row.get("review_count")) else 0,
            "is_open": is_open_b,
            "is_open_i": 1 if is_open_b else 0,
            "attributes": attributes_s,
            "bike_parking_i": int01(get_attr(row, "BikeParking")),
            "accepts_credit_cards_i": int01(get_attr(row, "BusinessAcceptsCreditCards")),
            "price_range": parse_int(get_attr(row, "RestaurantsPriceRange2")) or 0,
            "wifi": normalize_wifi(get_attr(row, "WiFi")) or "unknown",
            "parking_garage_i": int01(park.get("garage")),
            "parking_street_i": int01(park.get("street")),
            "parking_validated_i": int01(park.get("validated")),
            "parking_lot_i": int01(park.get("lot")),
            "parking_valet_i": int01(park.get("valet")),
            "wheelchair_accessible_i": int01(get_attr(row, "WheelchairAccessible")),
            "happy_hour_i": int01(get_attr(row, "HappyHour")),
            "outdoor_seating_i": int01(get_attr(row, "OutdoorSeating")),
            "has_tv_i": int01(get_attr(row, "HasTV")),
            "takes_out_i": int01(get_attr(row, "RestaurantsTakeOut")),
            "delivery_i": int01(get_attr(row, "RestaurantsDelivery")),
            "reservations_i": int01(get_attr(row, "RestaurantsReservations")),
            "dogs_allowed_i": int01(get_attr(row, "DogsAllowed")),
            "by_appointment_only_i": int01(get_attr(row, "ByAppointmentOnly")),
            "category_tags": cats,
            "categories_text": categories_text_from_list(cats),
            "hours": hours_s,
            "mon_open": mo_o,
            "mon_close": mo_c,
            "tue_open": tu_o,
            "tue_close": tu_c,
            "wed_open": we_o,
            "wed_close": we_c,
            "thu_open": th_o,
            "thu_close": th_c,
            "fri_open": fr_o,
            "fri_close": fr_c,
            "sat_open": sa_o,
            "sat_close": sa_c,
            "sun_open": su_o,
            "sun_close": su_c,
        }
    )


df_businesses_index = df_businesses.apply(business_row_for_index, axis=1)
parser = sl.DataFrameParser(business)
# Ingest with source.put([df_businesses_index]) — not the raw Yelp DataFrame.


# ----------------------------
# Example: categories text similarity space
# ----------------------------

categories_space = sl.TextSimilaritySpace(
    text=business.categories_text,
    model="sentence-transformers/all-MiniLM-L6-v2",
)


# NumberSpaces encode numeric input in special ways to reflect a relationship
# here we express relationships to price (lower the better), or ratings and review counts (more/higher the better)
review_count_space = sl.NumberSpace(number=business.review_count, mode=sl.Mode.MAXIMUM, scale=sl.LogarithmicScale(), min_value=5, max_value=1359)
review_rating_space = sl.NumberSpace(number=business.stars, mode=sl.Mode.MAXIMUM, min_value=1, max_value=5)
 
    

01:37:21 superlinked.framework.common.space.embedding.model_based.embedding_engine_manager INFO   Consider caching model dimension.


### Building the index

In [34]:
#Derive the index
# Create the space (index) by combining all the above spaces and feature mappings.
# `fields` order matches `Business` / `df_businesses_index` (same as df_businesses, minus IdField).

business_index = sl.Index(
    spaces=[
        categories_space,
        review_count_space,
        review_rating_space,
        # Add other spaces as needed if defined, like embedding or geo spaces
    ],
    fields=[
        business.name,
        business.address,
        business.city,
        business.state,
        business.postal_code,
        business.latitude,
        business.longitude,
        business.stars,
        business.review_count,
        business.is_open,
        business.is_open_i,
        business.attributes,
        business.bike_parking_i,
        business.accepts_credit_cards_i,
        business.price_range,
        business.wifi,
        business.parking_garage_i,
        business.parking_street_i,
        business.parking_validated_i,
        business.parking_lot_i,
        business.parking_valet_i,
        business.wheelchair_accessible_i,
        business.happy_hour_i,
        business.outdoor_seating_i,
        business.has_tv_i,
        business.takes_out_i,
        business.delivery_i,
        business.reservations_i,
        business.dogs_allowed_i,
        business.by_appointment_only_i,
        business.category_tags,
        business.categories_text,
        business.hours,
        business.mon_open,
        business.mon_close,
        business.tue_open,
        business.tue_close,
        business.wed_open,
        business.wed_close,
        business.thu_open,
        business.thu_close,
        business.fri_open,
        business.fri_close,
        business.sat_open,
        business.sat_close,
        business.sun_open,
        business.sun_close,
    ],
)

01:37:21 superlinked.framework.dsl.index.index INFO   initialized index


### Defining the main query

In [35]:
openai_config = sl.OpenAIClientConfig(
    api_key=config.openai_api_key,
    model="gpt-4o-mini",
)

In [36]:

query = (
    sl.Query(
        business_index,
        weights={
            review_count_space: sl.Param(
                "review_count_weight"
            ),
            review_rating_space: sl.Param(
                "rreview_rating_weight"
            ),
            categories_space: sl.Param("categories_weight"),
        },
    )
    .find(business)
    .similar(
        categories_space.text,
        sl.Param(
            "description",
            description=(
                "Semantic text for the business category field: city, neighborhood, vibe, atmosphere, "
                "dietary or service hints. Omit repeating the venue type; use required_category_tags for that."
            ),
        ),
    )
)

# We can specify number of retreved results like this:
query = query.limit(sl.Param("limit", default=4))

# We want all fields to be returned
query = query.select_all()

# .. and all the metadata including knn_params and partial_scores
query = query.include_metadata()

# Now let's add hard-filtering
# for city:
query = query.filter(
    business.city.in_(sl.Param("city", description="used to filter by city"))
)

# ... for numerical attributes:
query = (
    query
    .filter(business.stars >= sl.Param("min_rating"))
    .filter(business.stars <= sl.Param("max_rating"))
)

# ... and for all categorical attributes:
CategoryFilter = namedtuple(
    "CategoryFilter", ["operator", "param_name", "category_name", "description"]
)

filters = [
    CategoryFilter(
        operator=lambda p: business.category_tags.contains_all(p),
        param_name="required_category_tags",
        category_name="required_category_tags",
        description=(
            "List of Yelp category strings the business must have (e.g. ['Cocktail Bars'] or "
            "['Restaurants', 'Italian']). Match Yelp spelling/casing. Use null if the user does not name a venue type."
        ),
    ),
    CategoryFilter(
        operator=lambda p: business.is_open_i == p,
        param_name="is_open_i",
        category_name="is_open_i",
        description="Open status: 1 = open, 0 = closed. Use null if not specified.",
    ),
    CategoryFilter(
        operator=lambda p: business.bike_parking_i == p,
        param_name="bike_parking_i",
        category_name="bike_parking_i",
        description="1 = must have bike parking, 0 = must not, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.accepts_credit_cards_i == p,
        param_name="accepts_credit_cards_i",
        category_name="accepts_credit_cards_i",
        description="1 = accepts cards, 0 = does not, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.price_range == p,
        param_name="price_range",
        category_name="price_range",
        description="Yelp price level 1–4; use null if not specified.",
    ),
    CategoryFilter(
        operator=lambda p: business.wifi == p,
        param_name="wifi",
        category_name="wifi",
        description="WiFi: no, free, paid, or unknown; null if not specified.",
    ),
    CategoryFilter(
        operator=lambda p: business.parking_garage_i == p,
        param_name="parking_garage_i",
        category_name="parking_garage_i",
        description="1 = garage parking, 0 = no garage, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.parking_lot_i == p,
        param_name="parking_lot_i",
        category_name="parking_lot_i",
        description="1 = lot parking, 0 = no lot, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.parking_street_i == p,
        param_name="parking_street_i",
        category_name="parking_street_i",
        description="1 = street parking, 0 = no street, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.parking_valet_i == p,
        param_name="parking_valet_i",
        category_name="parking_valet_i",
        description="1 = valet, 0 = no valet, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.parking_validated_i == p,
        param_name="parking_validated_i",
        category_name="parking_validated_i",
        description="1 = validated parking, 0 = no, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.by_appointment_only_i == p,
        param_name="by_appointment_only_i",
        category_name="by_appointment_only_i",
        description="1 = by appointment only, 0 = walk-in ok, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.outdoor_seating_i == p,
        param_name="outdoor_seating_i",
        category_name="outdoor_seating_i",
        description="1 = outdoor seating, 0 = no, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.wheelchair_accessible_i == p,
        param_name="wheelchair_accessible_i",
        category_name="wheelchair_accessible_i",
        description="1 = wheelchair accessible, 0 = not, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.happy_hour_i == p,
        param_name="happy_hour_i",
        category_name="happy_hour_i",
        description="1 = has happy hour, 0 = no, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.takes_out_i == p,
        param_name="takes_out_i",
        category_name="takes_out_i",
        description="1 = takeout, 0 = no takeout, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.reservations_i == p,
        param_name="reservations_i",
        category_name="reservations_i",
        description="1 = takes reservations, 0 = no, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.delivery_i == p,
        param_name="delivery_i",
        category_name="delivery_i",
        description="1 = delivery, 0 = no delivery, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.has_tv_i == p,
        param_name="has_tv_i",
        category_name="has_tv_i",
        description="1 = has TV, 0 = no, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda p: business.dogs_allowed_i == p,
        param_name="dogs_allowed_i",
        category_name="dogs_allowed_i",
        description="1 = dogs allowed, 0 = not, null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.mon_open <= t,
            business.mon_close >= t,
        ],
        param_name="mon_time",
        category_name="mon_time",
        description="Minutes since midnight (0–1439) on Monday; business must be open at that time. Null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.tue_open <= t,
            business.tue_close >= t,
        ],
        param_name="tue_time",
        category_name="tue_time",
        description="Minutes since midnight on Tuesday; null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.wed_open <= t,
            business.wed_close >= t,
        ],
        param_name="wed_time",
        category_name="wed_time",
        description="Minutes since midnight on Wednesday; null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.thu_open <= t,
            business.thu_close >= t,
        ],
        param_name="thu_time",
        category_name="thu_time",
        description="Minutes since midnight on Thursday; null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.fri_open <= t,
            business.fri_close >= t,
        ],
        param_name="fri_time",
        category_name="fri_time",
        description="Minutes since midnight on Friday; null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.sat_open <= t,
            business.sat_close >= t,
        ],
        param_name="sat_time",
        category_name="sat_time",
        description="Minutes since midnight on Saturday; null = ignore.",
    ),
    CategoryFilter(
        operator=lambda t: [
            business.sun_open <= t,
            business.sun_close >= t,
        ],
        param_name="sun_time",
        category_name="sun_time",
        description="Minutes since midnight on Sunday; null = ignore.",
    ),
]

for filter_item in filters:
    param = sl.Param(
        filter_item.param_name,
        description=filter_item.description,
        # options= cat_options[filter_item.category_name],
    )

    ops = filter_item.operator(param)
    if isinstance(ops, list):
        for op in ops:
            query = query.filter(op)
    else:
        query = query.filter(ops)


system_prompt = (
    "Extract the search parameters from the user query.\n"
    "Advices:\n"
    "**required_category_tags** — When the user names a venue or business type, set this to a list of Yelp-style "
    "category labels that capture that type (e.g. ['Cocktail Bars'], ['Mexican'], ['Restaurants', 'Italian']). "
    "Every listed tag must appear on the business. Use common Yelp names and Title Case. Use null if they do not "
    "name a specific type.\n"
    "**description** — Free text for semantic match on stored category text: location, neighborhood, vibe, "
    "atmosphere, dietary notes. Do not repeat the venue-type wording here; that belongs in required_category_tags.\n"
    "**is_open_i** — 1 = open only, 0 = closed only, null = ignore.\n"
    "Params ending in _i are 0/1 amenity flags: 1 = must have, 0 = must not, null = ignore.\n"
    "**wifi** — one of: no, free, paid, unknown; null if not specified.\n"
    "**price_range** — Yelp dollar level 1–4; null if not specified.\n"
    "**mon_time** … **sun_time** — minutes since midnight (e.g. 720 = noon) for that weekday; null unless the user asks about being open at a specific time that day.\n"
    # "**'include' and 'exclude' attributes**\n"
    # "Use relevant amenities, for example, include 'Cot' when user mentions 'baby',"
    # "and exclude it when user mentions 'no children'.\n"
    # "If no amenities are mentioned, use None for 'include' and 'exclude'.\n"
    # "**'accomodation_type'**\n"
    # "If users searches for some restaurants, include 'Restaurants' in categories types, "
    # "same for other accomodation types.\n"
)

# And finally, let's add natural language interface on top
# that will call LLM to parse user natural query
# into structured superlinked query, i.e. suggest parameters values.
query = query.with_natural_query(
    natural_query=sl.Param("natural_query"),
    client_config=openai_config,
    system_prompt=system_prompt,
)


### Loading the data into memory

In [37]:
# We define the source type. In this case, `InMemorySource`
source = sl.InMemorySource(
    business,
    parser=parser
)

executor = sl.InMemoryExecutor(sources=[source], indices=[business_index])
app = executor.run()

01:37:21 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
01:37:21 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag
01:37:21 superlinked.framework.dsl.executor.interactive.interactive_executor INFO   started in-memory app


In [38]:
df_businesses.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,cHZ1FUswyko119EkV_9aNQ,Upper Crust Pizza,1909 E Grant Rd,Tucson,AZ,85719,32.250429,-110.943524,3.5,180,1,"{'BikeParking': 'True', 'RestaurantsPriceRange...","Salad, Pizza, Chicken Wings, Restaurants","{'Monday': '11:0-21:0', 'Tuesday': '11:0-21:0'..."
1,goTAL_3Ns-bsMgfGmKDq0A,Spitale's Deli & Catering,3309 Division St,Metairie,LA,70002,30.004793,-90.164386,4.5,53,1,"{'RestaurantsPriceRange2': '1', 'RestaurantsRe...","American (New), Restaurants, Local Flavor, Delis","{'Monday': '9:0-16:0', 'Tuesday': '10:30-20:0'..."
2,L1TzHZ8OXfVXTfyzidstzg,Snooze An AM Eatery,"2500 E Grant Rd, Ste 182",Tucson,AZ,85716,32.250337,-110.935414,4.5,253,1,"{'Caters': 'True', 'OutdoorSeating': 'True', '...","Breakfast & Brunch, American (Traditional), Sa...","{'Monday': '0:0-0:0', 'Tuesday': '6:30-14:30',..."
3,DfkO7MHuy2wxgeUZ2-o3BA,Subway,204 St Charles Ave,New Orleans,LA,70130,29.951985,-90.069679,3.0,8,0,"{'RestaurantsTakeOut': 'True', 'RestaurantsDel...","Restaurants, Sandwiches, Fast Food","{'Monday': '7:0-17:0', 'Tuesday': '7:0-17:0', ..."
4,ByKCIpZWHYUTuCJcoZ-ONQ,Giovanni's,6583 Pardall Rd,Goleta,CA,93117,34.412901,-119.857810,4.0,87,0,"{'RestaurantsTakeOut': 'True', 'RestaurantsGoo...","Salad, Sandwiches, Restaurants, Pizza, Italian","{'Monday': '11:0-20:0', 'Tuesday': '11:0-20:0'..."


In [39]:
source.put([df_businesses_index])

01:37:24 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
01:37:24 superlinked.framework.common.delayed_evaluator INFO   Processed vdb write


### Running queries

In [ ]:
results = app.query(
    query,
    natural_query="looking for open cocktail bars in New Orleans that has happy hour with outdoor seating and garage parking ",
    limit=5,
)

01:37:28 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
01:37:28 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
01:37:28 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


In [41]:
df_results = format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(results))
df_results


,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,...,thu_open,thu_close,fri_open,fri_close,sat_open,sat_close,sun_open,sun_close,id,similarity_score
0,Trenasse,"444 St Charles Ave, Ste 100, Intercontinental ...",New Orleans,LA,70130,29.950446,-90.069999,4.0,1059,True,...,11:00,22:00,11:00,22:00,11:00,22:00,10:00,22:00,uR7G8I4Cef9D9R340TN24Q,0.485116


In [42]:
df_businesses[df_businesses['business_id'] == 'uR7G8I4Cef9D9R340TN24Q']['attributes'].to_list()

[{'BusinessAcceptsCreditCards': 'True',
  'Alcohol': "'full_bar'",
  'RestaurantsTakeOut': 'True',
  'OutdoorSeating': 'True',
  'WiFi': "'free'",
  'RestaurantsTableService': 'True',
  'BikeParking': 'True',
  'RestaurantsReservations': 'True',
  'BusinessParking': "{'garage': True, 'street': True, 'validated': False, 'lot': False, 'valet': True}",
  'HasTV': 'True',
  'RestaurantsPriceRange2': '2',
  'WheelchairAccessible': 'True',
  'DogsAllowed': 'True',
  'NoiseLevel': "'average'",
  'GoodForKids': 'True',
  'RestaurantsDelivery': 'False',
  'Smoking': "u'no'",
  'RestaurantsGoodForGroups': 'True',
  'HappyHour': 'True',
  'ByAppointmentOnly': 'False',
  'RestaurantsAttire': "'casual'",
  'Caters': 'True',
  'Corkage': 'True',
  'Ambience': "{'touristy': False, 'hipster': False, 'romantic': False, 'divey': False, 'intimate': False, 'trendy': True, 'upscale': False, 'classy': True, 'casual': False}",
  'BestNights': "{u'monday': False, u'tuesday': False, u'wednesday': False, u'thur

In [43]:
df_businesses_index.columns.to_list()

['business_id',
 'name',
 'address',
 'city',
 'state',
 'postal_code',
 'latitude',
 'longitude',
 'stars',
 'review_count',
 'is_open',
 'is_open_i',
 'attributes',
 'bike_parking_i',
 'accepts_credit_cards_i',
 'price_range',
 'wifi',
 'parking_garage_i',
 'parking_street_i',
 'parking_validated_i',
 'parking_lot_i',
 'parking_valet_i',
 'wheelchair_accessible_i',
 'happy_hour_i',
 'outdoor_seating_i',
 'has_tv_i',
 'takes_out_i',
 'delivery_i',
 'reservations_i',
 'dogs_allowed_i',
 'by_appointment_only_i',
 'category_tags',
 'categories_text',
 'hours',
 'mon_open',
 'mon_close',
 'tue_open',
 'tue_close',
 'wed_open',
 'wed_close',
 'thu_open',
 'thu_close',
 'fri_open',
 'fri_close',
 'sat_open',
 'sat_close',
 'sun_open',
 'sun_close']

### Using Qdrant DB

Start Qdrant from the repo root: `make run-docker-compose`.
Superlinked can persist the same index to Qdrant by passing `sl.QdrantVectorDatabase` into an executor. Vector sizes are derived from the `TextSimilaritySpace` model.

**Qdrant collection name:** Superlinked uses **`APP_ID`** (default `"default"`). Configure it in **`config.yaml`** under `framework:` — this project sets `APP_ID: yelp-businesses-collection-00`. The file must be named **`config.yaml`** (Superlinked’s default) and live in the **process working directory** (repo root and `notebooks/` each have a copy). **Restart the kernel** after editing YAML.

**`RestExecutor`** expects each query as `sl.RestQuery(rest_descriptor=sl.RestDescriptor(query_path="..."), query_descriptor=query)` — not a bare `query` object (that is a `QueryDescriptor` and has no `.path` for routing).

Use a **separate** ingest source (`source_qdrant` below) so you do not double-register the in-memory `source` if you already ran `InMemoryExecutor` in this session. For notebooks, `sl.InteractiveExecutor` + `InMemorySource` is simpler; `RestSource` + `RestExecutor` is for HTTP-style ingest/search wiring.

In [44]:
qdrant_vdb = sl.QdrantVectorDatabase(
    url="http://localhost:6333",
    api_key="",  # empty for local Qdrant; use your cloud API key for Qdrant Cloud
)

source_qdrant = sl.RestSource(
    business,
    parser=parser,
)

# RestExecutor needs sl.RestQuery (path for /api/v1/search/<query_path> by default).
business_rest_query = sl.RestQuery(
    rest_descriptor=sl.RestDescriptor(query_path="business_search"),
    query_descriptor=query,
)

executor_qdrant = sl.RestExecutor(
    sources=[source_qdrant],
    indices=[business_index],
    vector_database=qdrant_vdb,
    queries=[business_rest_query],
)
qdrant_app = executor_qdrant.run()

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:87: UserWarning: Api key is used with an insecure connection.
  self._client = AsyncQdrantClient(
/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:92: UserWarning: Api key is used with an insecure connection.
  self._sync_client = QdrantClient(


01:37:29 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
01:37:29 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag


In [45]:
source_qdrant.put([df_businesses_index])

01:37:32 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
01:37:33 superlinked.framework.common.delayed_evaluator INFO   Processed vdb write


In [46]:
qdrant_results = qdrant_app.query(
    query,
    natural_query="looking for open cocktail bars in new orleans that has happy hour with outdoor seating and garage parking ",
    limit=5,
)
format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))

01:37:39 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
01:37:39 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
01:37:39 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,...,thu_open,thu_close,fri_open,fri_close,sat_open,sat_close,sun_open,sun_close,id,similarity_score
0,Trenasse,"444 St Charles Ave, Ste 100, Intercontinental ...",New Orleans,LA,70130,29.950446,-90.069999,4.0,1059,True,...,11:00,22:00,11:00,22:00,11:00,22:00,10:00,22:00,uR7G8I4Cef9D9R340TN24Q,0.485135


In [47]:
def _result_to_restaurants(result) -> list[dict[str, Any]]:
    df = sl.PandasConverter.to_pandas(result).rename(columns={"id": "business_id"})
    df = df.assign(
        categories=df.get("category_tags", df.get("categories_text")),
        is_open=df["is_open"].astype(int),
    )
    for c in ("attributes", "hours"):
        df[c] = df[c].map(
            lambda v: json.loads(v) if isinstance(v, str) and v.strip()
            else ({} if v in ("", None) else v)
        )
    return df.reindex(columns=df_businesses.columns).to_dict(orient="records")

In [48]:
preprocessed_context=_result_to_restaurants(qdrant_results)
preprocessed_context

[{'business_id': 'uR7G8I4Cef9D9R340TN24Q',
  'name': 'Trenasse',
  'address': '444 St Charles Ave, Ste 100, Intercontinental Hotel',
  'city': 'New Orleans',
  'state': 'LA',
  'postal_code': '70130',
  'latitude': 29.9504461383,
  'longitude': -90.0699986056,
  'stars': 4.0,
  'review_count': 1059,
  'is_open': 1,
  'attributes': {'Alcohol': "'full_bar'",
   'Ambience': "{'touristy': False, 'hipster': False, 'romantic': False, 'divey': False, 'intimate': False, 'trendy': True, 'upscale': False, 'classy': True, 'casual': False}",
   'BestNights': "{u'monday': False, u'tuesday': False, u'wednesday': False, u'thursday': False, u'friday': False, u'saturday': False, u'sunday': False}",
   'BikeParking': 'True',
   'BusinessAcceptsCreditCards': 'True',
   'BusinessParking': "{'garage': True, 'street': True, 'validated': False, 'lot': False, 'valet': True}",
   'ByAppointmentOnly': 'False',
   'Caters': 'True',
   'Corkage': 'True',
   'DogsAllowed': 'True',
   'GoodForKids': 'True',
   'Goo

In [49]:
def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
    - respond naturally and provide as much details as possible to the user request 
    - Refrain from using filter sush as is_open =True ...Rather say open today

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt

In [ ]:
prompt=build_prompt(preprocessed_context, "looking for open cocktail bars in New Orleans that has happy hour with outdoor seating and garage parking that is open before 10")
print(prompt)


    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
    - respond naturally and provide as much details as possible to the user request 
    - Refrain from using filter sush as is_open =True ...Rather say open today

    Context:
    [{'business_id': 'uR7G8I4Cef9D9R340TN24Q', 'name': 'Trenasse', 'address': '444 St Charles Ave, Ste 100, Intercontinental Hotel', 'city': 'New Orleans', 'state': 'LA', 'postal_code': '70130', 'latitude': 29.9504461383, 'longitude': -90.0699986056, 'stars': 4.0, 'review_count': 1059, 'is_open': 1, 'attributes': {'Alcohol': "'full_bar'", 'Ambience': "{'touristy': False, 'hipster': False, 'romantic': False, 'divey': False, 'intimate': False, 'trendy': True, 'upscale': False, 'classy': True, 'casua

In [51]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role":"system", "content": prompt}],
        reasoning_effort="medium"
    )
    return response.choices[0].message.content

In [52]:
print(generate_answer(prompt))

Great news — Trenasse in New Orleans fits all your criteria.

What it offers:
- Name: Trenasse
- Address: 444 St Charles Ave, Ste 100, Intercontinental Hotel, New Orleans, LA 70130
- Open now: Yes (open today; hours listed below)
- Type: Cocktail bar (full bar)
- Happy hour: Yes
- Outdoor seating: Yes
- Parking: Garage parking available (also street parking; valet OK)
- Reservations: Accepted
- Hours: Mon–Sat 11:00–22:00; Sun 10:00–22:00
- Price range: Moderate
- Other notes: Wheelchair accessible, free WiFi, good for groups, casual but upscale vibe, noise level average

If you’d like, I can pull up directions, or help you book a reservation at Trenasse. I can also look for more open cocktail bars in New Orleans with outdoor seating and garage parking if you want a few more options.


In [55]:
def Retrieve_context(question, qdrant_app):
    qdrant_results = qdrant_app.query(
    query,
    natural_query=question,
    limit=5,
)
    format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))
    return qdrant_results


In [56]:
def RAG_pipeline(question):
    context=Retrieve_context(question, qdrant_app)
    preprocessed_context=_result_to_restaurants(context)
    prompt=build_prompt(preprocessed_context, question)
    answer=generate_answer(prompt)
    return answer

In [57]:
print(RAG_pipeline("looking for open cocktail bars in New Orleans that has happy hour with outdoor seating and garage parking that is open before 10"))

01:49:15 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
01:49:15 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
01:49:15 superlinked.framework.dsl.executor.query.query_executor INFO   executed query
Here’s what I found among the available options in New Orleans:

- Trenasse (Intercontinental Hotel, 444 St Charles Ave)
  - Type: Cocktail bar with a strong nightlife vibe
  - Open hours: Sunday 10:00–22:00; Monday–Saturday 11:00–22:00
  - Open before 10:00: No. The earliest opening is 10:00 a.m. (only on Sundays); other days start at 11:00 a.m.
  - Happy hour: Yes
  - Outdoor seating: Yes
  - Parking: Garage parking available
  - Other notes: Full bar, casual attire, good for groups, reservations accepted, takeout available, wheelchair accessible, free WiFi

Bottom line: There isn’t a listed option that is open before 10:00 a.m. Trenasse best fits your other criteria (cocktail bar, happy hour, outdoor se